# 06 — India Layer: Synthetic Borrower Profiles
**TrustGraph Credit Risk Project**

> **⚠ ALL DATA IN THIS NOTEBOOK IS SYNTHETIC AND SIMULATED.**  
> No real borrower, transaction, or credit record is used.  
> Economic parameters are loosely calibrated to publicly available RBI/MSME aggregate statistics  
> but are NOT precise reproductions of those statistics.  
> This notebook exists solely for model-testing and research purposes.

Generates 3 synthetic Indian borrower profiles, runs them through both the production
LightGBM model and the WOE scorecard, and produces an approve/refer/decline decision table.

In [1]:
import os, sys, pickle, warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')

ROOT       = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
DATA_PROC  = ROOT / 'data' / 'processed'
MODELS_DIR = ROOT / 'models'
SRC_DIR    = ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

from india_layer import (
    generate_transactions, map_to_trustgraph_schema, make_decision
)

# Verify model artifacts exist
lgbm_path = MODELS_DIR / 'lgbm_final.pkl'
sc_path   = MODELS_DIR / 'scorecard_final.pkl'
for p in [lgbm_path, sc_path]:
    if not p.exists():
        raise FileNotFoundError(f"{p} — run notebooks 04 and 05 first.")
print("Models loaded OK")

Models loaded OK


## 1. Load saved models

In [2]:
with open(lgbm_path, 'rb') as f:
    lgbm_model = pickle.load(f)

with open(sc_path, 'rb') as f:
    sc_bundle = pickle.load(f)
sc_lr_model       = sc_bundle['lr_model']
sc_woe_pipeline   = sc_bundle['woe_pipeline']
sc_features       = sc_bundle['features']
sc_woe_feat_names = sc_bundle['woe_feature_names']

lgbm_features = lgbm_model.feature_name_
print(f"LightGBM features: {len(lgbm_features)}")
print(f"Scorecard features: {sc_features}")

LightGBM features: 128
Scorecard features: ['EXT_SOURCE_MEAN', 'EXT_SOURCE_MIN', 'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_TERM', 'DAYS_EMPLOYED_RATIO', 'AMT_CREDIT', 'AMT_INCOME_TOTAL', 'DAYS_BIRTH', 'FLAG_MISSING_EXT']


## 2. Generate 12-month synthetic transactions for each profile

> **SYNTHETIC DATA** — all transactions below are algorithmically generated.

In [3]:
profiles = {
    'Kirana Shop Owner':    'kirana',
    'Gig Worker (Swiggy/Ola)': 'gig',
    'Small Farmer':         'farmer',
}

txn_data = {}
for label, ptype in profiles.items():
    txn_data[label] = generate_transactions(ptype, months=12)
    print(f"\n{'='*55}")
    print(f"  {label}  [SYNTHETIC]")
    print(f"{'='*55}")
    print(txn_data[label].to_string(index=False))
    print(f"  Annual income (avg*12): INR {txn_data[label]['income_inr'].mean()*12:,.0f}")


  Kirana Shop Owner  [SYNTHETIC]
     month  income_inr  expenses_inr  net_cashflow_inr
2023-01-01    42639.91      24678.83          17961.08
2023-02-01    40612.35      24958.42          15653.94
2023-03-01    45754.74      24647.30          21107.45
2023-04-01    45294.44      22441.03          22853.41
2023-05-01    38281.85      21472.94          16808.91
2023-06-01    32197.65      14899.89          17297.75
2023-07-01    32969.40      20293.53          12675.87
2023-08-01    33068.71      19058.59          14010.13
2023-09-01    39866.48      23984.37          15882.11
2023-10-01    47446.16      24714.95          22731.21
2023-11-01    53493.02      34456.97          19036.05
2023-12-01    47124.03      29623.31          17500.72
  Annual income (avg*12): INR 498,749

  Gig Worker (Swiggy/Ola)  [SYNTHETIC]
     month  income_inr  expenses_inr  net_cashflow_inr
2023-01-01    22458.02      10689.66          11768.35
2023-02-01    22732.32      12392.32          10340.00
2023-03-

## 3. Map cash-flow profiles to TrustGraph feature schema

> **SYNTHETIC** — feature values are derived from simulated data, NOT real credit applications.

In [4]:
loan_configs = {
    'Kirana Shop Owner':       {'loan_amount_inr': 200_000, 'loan_term_months': 36},
    'Gig Worker (Swiggy/Ola)': {'loan_amount_inr': 100_000, 'loan_term_months': 24},
    'Small Farmer':            {'loan_amount_inr': 150_000, 'loan_term_months': 48},
}

feature_rows = {}
for label, ptype in profiles.items():
    feature_rows[label] = map_to_trustgraph_schema(
        txn_df=txn_data[label],
        profile_type=ptype,
        **loan_configs[label],
    )

profile_df = pd.DataFrame(feature_rows).T
display_cols = [
    'AMT_CREDIT','AMT_INCOME_TOTAL','CREDIT_INCOME_RATIO',
    'ANNUITY_INCOME_RATIO','CREDIT_TERM','DAYS_EMPLOYED_RATIO',
    'EXT_SOURCE_MEAN','FLAG_MISSING_EXT','INCOME_PER_PERSON',
    '_avg_monthly_income','_income_volatility',
]
print("[SYNTHETIC FEATURES]")
profile_df[display_cols].astype(float, errors='ignore').round(4)

[SYNTHETIC FEATURES]


,AMT_CREDIT,AMT_INCOME_TOTAL,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,CREDIT_TERM,DAYS_EMPLOYED_RATIO,EXT_SOURCE_MEAN,FLAG_MISSING_EXT,INCOME_PER_PERSON,_avg_monthly_income,_income_volatility
Kirana Shop Owner,200000.0,498748.74,0.4010,0.1337,36.0,0.2083,NaN,1.0,124687.185,41562.3950,0.1604
Gig Worker (Swiggy/Ola),100000.0,299214.07,0.3342,0.1671,24.0,0.1200,NaN,1.0,149607.035,24934.5058,0.1200
Small Farmer,150000.0,285875.36,0.5247,0.1312,48.0,0.3425,NaN,1.0,57175.072,23822.9467,2.2229


## 4. Score each profile through LightGBM and Scorecard

> **SYNTHETIC SCORING** — scores are outputs on simulated feature vectors, not real risk assessments.

In [5]:
import sys
sys.path.insert(0, str(SRC_DIR))

def prep_for_lgbm(row_dict: dict, feature_names: list) -> pd.DataFrame:
    """
    Build a single-row DataFrame aligned to the LightGBM feature list.
    Missing columns are filled with 0.
    SYNTHETIC — input is derived from simulated profile data.
    """
    row = {f: row_dict.get(f, 0) for f in feature_names}
    return pd.DataFrame([row])


def woe_transform_single(series_val, woe_table, breakpoints):
    """
    Apply WOE lookup for a single scalar value.
    SYNTHETIC — used only on simulated profile data.
    """
    import numpy as np
    if pd.isna(series_val):
        miss = woe_table[woe_table['Bin'] == 'MISSING']['WoE']
        return float(miss.iloc[0]) if len(miss) else 0.0
    non_miss = woe_table[woe_table['Bin'] != 'MISSING']
    bin_idx = int(np.searchsorted(breakpoints[1:-1], series_val, side='right'))
    if bin_idx < len(non_miss):
        return float(non_miss.iloc[bin_idx]['WoE'])
    return float(non_miss.iloc[-1]['WoE']) if len(non_miss) else 0.0


def prep_and_score_scorecard(row_dict, woe_pipeline, sc_features, lr_model):
    """
    WOE-transform a single row dict and return default probability.
    SYNTHETIC — input is derived from simulated profile data.
    """
    woe_row = {}
    for feat in sc_features:
        info = woe_pipeline[feat]
        val  = row_dict.get(feat, np.nan)
        woe_row[f'{feat}_WOE'] = woe_transform_single(val, info['table'], info['breakpoints'])
    X_woe = pd.DataFrame([woe_row])
    return float(lr_model.predict_proba(X_woe)[0, 1])


results = []
for label, ptype in profiles.items():
    row = feature_rows[label]

    # LightGBM score
    X_lgbm    = prep_for_lgbm(row, lgbm_features)
    lgbm_prob = float(lgbm_model.predict_proba(X_lgbm)[0, 1])

    # Scorecard score (manual WOE pipeline)
    sc_prob = prep_and_score_scorecard(row, sc_woe_pipeline, sc_features, sc_lr_model)

    results.append({
        'Borrower Type [SYNTHETIC]': label,
        'Avg Monthly Income (INR)':  f"₹{row['_avg_monthly_income']:,.0f}",
        'Loan Amount (INR)':         f"₹{row['AMT_CREDIT']:,.0f}",
        'LightGBM Default Prob':     round(lgbm_prob, 4),
        'LightGBM Decision':         make_decision(lgbm_prob),
        'Scorecard Default Prob':    round(sc_prob, 4),
        'Scorecard Decision':        make_decision(sc_prob),
    })

result_df = pd.DataFrame(results)
print("[SYNTHETIC SCORING RESULTS]")
result_df

[SYNTHETIC SCORING RESULTS]


,Borrower Type [SYNTHETIC],Avg Monthly Income (INR),Loan Amount (INR),LightGBM Default Prob,LightGBM Decision,Scorecard Default Prob,Scorecard Decision
0,Kirana Shop Owner,"₹41,562","₹200,000",0.6571,Decline,0.4287,Refer
1,Gig Worker (Swiggy/Ola),"₹24,935","₹100,000",0.5617,Decline,0.3613,Refer
2,Small Farmer,"₹23,823","₹150,000",0.4906,Refer,0.3463,Refer


## 5. Save profiles and results

> Files saved are marked **SYNTHETIC** — not real credit data.

In [6]:
# Full feature matrix for downstream use (SHAP in notebook 07)
save_df = profile_df.copy()
save_df.index.name = 'borrower_type'
save_df['_lgbm_default_prob'] = [r['LightGBM Default Prob'] for r in results]
save_df['_lgbm_decision']     = [r['LightGBM Decision']     for r in results]
save_df['_sc_default_prob']   = [r['Scorecard Default Prob'] for r in results]
save_df['_sc_decision']       = [r['Scorecard Decision']     for r in results]
save_df['_data_is_synthetic'] = True

profiles_path = DATA_PROC / 'india_profiles.csv'
save_df.reset_index().to_csv(profiles_path, index=False)
print(f"Saved: {profiles_path}")

# Clean results table
result_df.to_csv(DATA_PROC / 'india_decisions.csv', index=False)
print(f"Saved: {DATA_PROC / 'india_decisions.csv'}")

Saved: C:\Users\krish\Downloads\TrustGraph\data\processed\india_profiles.csv


Saved: C:\Users\krish\Downloads\TrustGraph\data\processed\india_decisions.csv


## 6. Final decision table

> **REMINDER: All borrower profiles, income figures, and model scores below are SYNTHETIC.**

In [7]:
print("=" * 74)
print("  TRUSTGRAPH — INDIA PROFILES SCORING  [ALL DATA SYNTHETIC]")
print("=" * 74)
for r in results:
    print(f"\n  {r['Borrower Type [SYNTHETIC]']}")
    print(f"    Income: {r['Avg Monthly Income (INR)']} / month (synthetic)")
    print(f"    Loan:   {r['Loan Amount (INR)']}")
    print(f"    LightGBM → P(default)={r['LightGBM Default Prob']:.4f}  Decision: {r['LightGBM Decision']}")
    print(f"    Scorecard→ P(default)={r['Scorecard Default Prob']:.4f}  Decision: {r['Scorecard Decision']}")
print("\n" + "=" * 74)

  TRUSTGRAPH — INDIA PROFILES SCORING  [ALL DATA SYNTHETIC]

  Kirana Shop Owner
    Income: ₹41,562 / month (synthetic)
    Loan:   ₹200,000
    LightGBM → P(default)=0.6571  Decision: Decline
    Scorecard→ P(default)=0.4287  Decision: Refer

  Gig Worker (Swiggy/Ola)
    Income: ₹24,935 / month (synthetic)
    Loan:   ₹100,000
    LightGBM → P(default)=0.5617  Decision: Decline
    Scorecard→ P(default)=0.3613  Decision: Refer

  Small Farmer
    Income: ₹23,823 / month (synthetic)
    Loan:   ₹150,000
    LightGBM → P(default)=0.4906  Decision: Refer
    Scorecard→ P(default)=0.3463  Decision: Refer

